# Simulated effective stage under temperature-shift protocols

Predicts the **expected effective developmental stage** (28C-equivalent hpf) for embryo cohorts under a three-phase temperature-shift protocol, using the **empirical** temperature-dependent developmental rate observed in the hotfish data — *not* the Kimmel `0.055T−0.57` formula.

### Protocol
- **Phase 1** (0 → 6 h): all cohorts at **28C**.
- **Phase 2** (6 → 22 h real time): shifted to treatment **T ∈ {19, 24, 32, 34, 35}C**.
- **Phase 3** (22 h → end): shifted **back to 28C**. The effective stage *at* the 22 h shift-back differs per cohort (hotter/faster cohorts are developmentally older) — that divergence is the point.

### Empirical rate (from data, not Kimmel)
Hotfish embryos sat at their treatment temperature continuously from ~6 hpf to collection (24/30/36 h). Fitting `stage = s6 + rate_emp(T)·(t−6)` with a shared 6h stage `s6` and per-temperature rate gives the empirical developmental rate. It is **non-monotonic**: rising to a peak near 32C then falling at 33.5/35C (hot-end breakdown). A degree-3 `rate_emp(T)` captures that shape and supplies the interpolated 34C and 20C values. `28C-equivalent` because the rate is normalized in stage-hpf per real-hour, and the 28C rate ≈ 0.77.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

import morph_stage_reliability_utils as mr
import shift_simulation_utils as sim

plt.style.use('default')
mpl.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white',
                     'axes.grid': True, 'grid.color': '#e6e6e6', 'grid.linewidth': 0.6,
                     'font.size': 10, 'savefig.bbox': 'tight'})

TREATMENT_TEMPS = [19, 24, 32, 34, 35]
SHIFTBACK_H = 22.0
FIG_DIR = mr.fig_dir()

# temperature colour map (diverging, centred on the 28C baseline)
tnorm = mpl.colors.TwoSlopeNorm(vmin=18, vcenter=28, vmax=36)
tcmap = plt.get_cmap('RdBu_r')
tcol = {T: tcmap(tnorm(T)) for T in TREATMENT_TEMPS + [28]}

## The empirical developmental-rate curve

Measured per-temperature rates (points) and the degree-3 fit (line) that drives the simulation. The Kimmel linear rate is shown for contrast — it misses the hot-end falloff entirely.

In [ ]:
fit = sim.fit_empirical_rate(degree=3)
print('shared stage at 6h (s6):', round(fit['s6'], 2), 'hpf')

Tgrid = np.linspace(18, 36, 200)
fig, ax = plt.subplots(figsize=(6.4, 4.4))
ax.plot(Tgrid, fit['rate_poly'](Tgrid), '-', color='#333333', lw=1.8, label='empirical rate (deg-3 fit)')
ax.plot(Tgrid, 0.055 * Tgrid - 0.57, '--', color='#999999', lw=1.2, label='Kimmel 0.055T−0.57')
for T, r in fit['rate_meas'].items():
    ax.scatter([T], [r], s=70, color=tcmap(tnorm(T)), edgecolor='black', linewidth=0.5, zorder=3)
for T in TREATMENT_TEMPS:
    ax.scatter([T], [fit['rate_fn'](T)], s=90, marker='D', facecolor='none',
               edgecolor='black', linewidth=1.3, zorder=4)
ax.set_xlabel('temperature (C)'); ax.set_ylabel('developmental rate (stage-hpf / real-hour)')
ax.set_title('Empirical developmental rate vs temperature\n(circles = measured, ◇ = simulated treatments)')
ax.legend(frameon=False, fontsize=8)
fig.savefig(FIG_DIR / 'S1_empirical_rate_curve.png', dpi=200); fig.savefig(FIG_DIR / 'S1_empirical_rate_curve.pdf')
plt.show()

print('rate at treatment temps:', {T: round(fit['rate_fn'](T), 3) for T in TREATMENT_TEMPS + [28]})

## Effective-stage trajectories

Effective (28C-equivalent) stage vs real clock time. Grey band = the 6→22 h treatment window; dashed line = the 22 h shift-back. During treatment the cohorts fan out; after shift-back all resume the 28C rate and run parallel, locking in the accumulated offset.

In [ ]:
traj, meta = sim.simulate_cohorts(TREATMENT_TEMPS, fit, shiftback_h=SHIFTBACK_H, t_max=40)

fig, ax = plt.subplots(figsize=(8.2, 5.2))
ax.axvspan(sim.BASELINE_START_H, SHIFTBACK_H, color='#f0f0f0', zorder=0, label='treatment window')
ax.axvline(SHIFTBACK_H, color='#888888', ls='--', lw=1)
# 28C reference (no shift)
ref_traj, _ = sim.simulate_cohorts([28], fit, shiftback_h=SHIFTBACK_H, t_max=40)
ax.plot(ref_traj['real_time_h'], ref_traj['eff_stage'], color='#444444', lw=1.2, ls=':', label='28C (no shift)')
for T in TREATMENT_TEMPS:
    d = traj[traj['treatment_T'] == T]
    ax.plot(d['real_time_h'], d['eff_stage'], color=tcol[T], lw=2, label=f'{T}C')
    ax.scatter([SHIFTBACK_H], [meta[T]['stage_at_shiftback']], color=tcol[T],
               s=45, edgecolor='black', linewidth=0.5, zorder=5)
ax.set_xlabel('real time (h)'); ax.set_ylabel('effective stage (28C-equivalent hpf)')
ax.set_title('Effective developmental stage under 6→22 h temperature shift, then back to 28C')
ax.legend(frameon=False, fontsize=8, ncol=2, loc='upper left')
fig.savefig(FIG_DIR / 'S2_effective_stage_trajectories.png', dpi=200); fig.savefig(FIG_DIR / 'S2_effective_stage_trajectories.pdf')
plt.show()

## Effective stage at the 22 h shift-back ("stage varies")

The developmental stage each cohort has reached at the moment of shift-back. Non-monotonic in temperature: 32C is most advanced; 34C and 35C are *less* advanced despite being hotter, because the empirical rate peaks at ~32C and declines beyond it.

In [ ]:
sb = pd.DataFrame({
    'treatment_T': TREATMENT_TEMPS,
    'rate_during_treatment': [round(meta[T]['rate_treatment'], 3) for T in TREATMENT_TEMPS],
    'eff_stage_at_22h': [round(meta[T]['stage_at_shiftback'], 2) for T in TREATMENT_TEMPS],
})
sb['stage_vs_28C_control'] = (sb['eff_stage_at_22h']
                              - float(ref_traj.iloc[(ref_traj['real_time_h'] - SHIFTBACK_H).abs().argmin()]['eff_stage'])).round(2)
print('Effective stage at the 22 h shift-back:')
sb

## Effective stage table at key real-times (20 → 36 h)

In [ ]:
table = sim.stage_at_times(traj, [20, 22, 24, 30, 36])
# add the 28C no-shift control row
ctrl = sim.stage_at_times(ref_traj, [20, 22, 24, 30, 36]); ctrl['treatment_T'] = 28
table = pd.concat([ctrl, table], ignore_index=True).sort_values('treatment_T').reset_index(drop=True)
print('Effective (28C-equivalent) stage at each real clock time (hpf):')
table

## Do shifted embryos match non-shifted staging at collection? (equivalent developmental age)

The key question: at each collection time (24/30/36 h), does a **shifted** cohort's effective stage match a **non-shifted** reference — and if not, *which* reference time does it resemble (a shifted-24h embryo might look like a standard-30h one)?

For each shifted collection point we compute the **equivalent reference age**: the clock time at which each reference reaches the shifted cohort's effective stage. Two references:
- **always-28C** (normal rearing) — does the pulse leave a permanent stage offset?
- **always-at-T** (continuous exposure = the actual measured hotfish cohorts) — does a 16 h pulse look like continuous exposure?

Equivalent age **>** collection time → shifted embryo looks developmentally **older** than that reference at the same clock time (and vice versa). NaN = shifted stage is outside the reference's achievable range (e.g. cold references never reach it)."

In [ ]:
COLLECTION_TIMES = [24, 30, 36]
eqt = sim.equivalent_age_table(TREATMENT_TEMPS, fit, collection_times=COLLECTION_TIMES,
                               shiftback_h=SHIFTBACK_H, t_max=60)

# reference trajectories over a wide time grid (for inversion + plotting)
t_ref = np.linspace(0, 50, 1001)
ref28 = sim.always_at_temperature_trajectory(28, fit, t_grid=t_ref)

fig, axes = plt.subplots(1, len(TREATMENT_TEMPS), figsize=(3.5 * len(TREATMENT_TEMPS), 4.4),
                         sharex=True, sharey=True)
for ax, T in zip(axes, TREATMENT_TEMPS):
    refT = sim.always_at_temperature_trajectory(T, fit, t_grid=t_ref)
    shifted = sim.effective_stage_trajectory(T, fit['rate_fn'], s6=fit['s6'], t_grid=t_ref,
                                             shiftback_h=SHIFTBACK_H)
    # reference curves
    ax.plot(t_ref, ref28, color='#444444', ls=':', lw=1.4, label='always 28C')
    ax.plot(t_ref, refT, color=tcol[T], ls='-', lw=1.4, alpha=0.6, label=f'always {T}C')
    ax.plot(t_ref, shifted, color=tcol[T], ls='-', lw=2.4, label=f'{T}C pulse→28C')
    ax.axvspan(sim.BASELINE_START_H, SHIFTBACK_H, color='#f4f4f4', zorder=0)
    # collection points + equivalent-age connectors
    for ct in COLLECTION_TIMES:
        row = eqt[(eqt.treatment_T == T) & (eqt.collection_h == ct)].iloc[0]
        s = row['shifted_stage']
        ax.scatter([ct], [s], color=tcol[T], s=42, edgecolor='black', linewidth=0.5, zorder=6)
        # horizontal line to where 28C reference reaches the same stage
        if np.isfinite(row['equiv_age_28C']):
            ax.plot([ct, row['equiv_age_28C']], [s, s], color='#444444', lw=0.8, ls='--', zorder=4)
            ax.scatter([row['equiv_age_28C']], [s], color='#444444', s=20, zorder=5)
    ax.set_title(f'{T}C pulse'); ax.set_xlabel('real time (h)')
    ax.set_xticks(COLLECTION_TIMES)
axes[0].set_ylabel('effective stage (28C-equiv hpf)')
axes[0].legend(frameon=False, fontsize=7, loc='upper left')
fig.suptitle('Do shifted embryos match non-shifted staging at collection?  '
             '(dashed = 28C-equivalent developmental age)', y=1.03)
fig.tight_layout()
fig.savefig(FIG_DIR / 'S3_equivalent_age_by_cohort.png', dpi=200); fig.savefig(FIG_DIR / 'S3_equivalent_age_by_cohort.pdf')
plt.show()

In [ ]:
# Equivalent-age table + a compact 'offset from collection time' summary panel.
eqt_show = eqt.copy()
eqt_show['off_vs_28C_age(h)'] = (eqt_show['equiv_age_28C'] - eqt_show['collection_h']).round(2)
eqt_show['off_vs_ownT_age(h)'] = (eqt_show['equiv_age_ownT'] - eqt_show['collection_h']).round(2)
print('Equivalent developmental age of shifted cohorts vs each reference:')
print('  (equiv_age > collection_h  =>  looks OLDER than that reference at collection)')
display(eqt_show[['treatment_T', 'collection_h', 'shifted_stage',
                  'equiv_age_28C', 'off_vs_28C_age(h)',
                  'equiv_age_ownT', 'off_vs_ownT_age(h)']])

# summary bars: developmental-age offset from the always-28C reference
fig, ax = plt.subplots(figsize=(8.4, 4.2))
x = np.arange(len(COLLECTION_TIMES)); w = 0.16
for i, T in enumerate(TREATMENT_TEMPS):
    d = eqt_show[eqt_show.treatment_T == T].set_index('collection_h').loc[COLLECTION_TIMES]
    ax.bar(x + (i - 2) * w, d['off_vs_28C_age(h)'], w, color=tcol[T], label=f'{T}C',
           edgecolor='black', linewidth=0.3)
ax.axhline(0, color='#333333', lw=0.8)
ax.set_xticks(x); ax.set_xticklabels([f'{c} h' for c in COLLECTION_TIMES])
ax.set_xlabel('collection time'); ax.set_ylabel('developmental-age offset vs always-28C (h)')
ax.set_title('How far off a normal 28C embryo does each shifted cohort look at collection?\n'
             '(+ = looks older/advanced, − = looks younger/delayed)')
ax.legend(frameon=False, fontsize=8, ncol=5, loc='upper center')
fig.savefig(FIG_DIR / 'S4_age_offset_vs_28C.png', dpi=200); fig.savefig(FIG_DIR / 'S4_age_offset_vs_28C.pdf')
plt.show()

### Notes & assumptions

- **Empirical rate**, not Kimmel: derived from hotfish `mdl_stage_hpf` cohort means via a shared-6h-baseline joint fit; degree-3 `rate_emp(T)`. 34C is interpolated between measured 33.5/35C; 20C is a mild extrapolation just above measured 19C.
- **Effective stage** = accumulated `∫ rate_emp(T(τ)) dτ`, anchored so all cohorts share stage `s6` at 6 h (they are identical through phase 1). Units are 28C-equivalent hpf (rate(28C) ≈ 0.77).
- **Shift-back is at 22 h real time** for all cohorts, so the stage reached at shift-back varies by cohort.
- **Hot-end caveat**: the 33.5/35C rates reflect embryos that were at those temperatures *continuously from 6 h*; a cohort only briefly exposed (6→22 h then rescued) may not accrue the same rate depression. This simulation assumes the steady-state empirical rate applies throughout the exposure — reasonable as a first pass, but the true transient response to a 16 h heat pulse could differ.
- **Post-shift** all cohorts resume the 28C rate; trajectories run parallel thereafter, so inter-cohort stage offsets established during treatment persist.